In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
%run ./UDF/udf_silver_incremental_ingest

In [0]:


src_bronze_path = "/Volumes/data_governance/bronze_lineage_analysis/lineage_table_level"
tgt_silver_table = "data_governance.silver_lineage_analysis.lineage_table_level"



In [0]:
df=silver_incremental_ingest(src_bronze_path,tgt_silver_table)

In [0]:
if df.count()==0:
    dbutils.notebook.exit("No new records to load")
else:
    pass

In [0]:
df.limit(20).display()

In [0]:

df = df.withColumn("entity_type", upper(trim(col("entity_type")))) \
       .withColumn("source_type", upper(trim(col("source_type")))) \
       .withColumn("target_type", upper(trim(col("target_type")))) \
       .withColumn("created_by", lower(trim(col("created_by"))))


df = df.withColumn(
    "event_timestamp",
    to_timestamp("event_time")
)

df = df.withColumn("event_year", year("event_timestamp")) \
       .withColumn("event_month", month("event_timestamp")) \
       .withColumn("event_day", dayofmonth("event_timestamp")) \
       .withColumn("event_hour", hour("event_timestamp"))


df = df.withColumn(
    "source_object",
    coalesce(col("source_table_full_name"), col("source_path"))
)

df = df.withColumn(
    "target_object",
    coalesce(col("target_table_full_name"), col("target_path"))
)

df = df.withColumn("job_id", col("entity_metadata.job_info.job_id")) \
       .withColumn("job_run_id", col("entity_metadata.job_info.job_run_id")) \
       .withColumn("notebook_id", col("entity_metadata.notebook_id")) \
       .withColumn("dashboard_id", col("entity_metadata.dashboard_id")) \
       .withColumn("sql_query_id", col("entity_metadata.sql_query_id"))


df = df.withColumn(
    "execution_type",
    when(col("entity_type") == "NOTEBOOK", "INTERACTIVE")
    .when(col("entity_type") == "JOB", "SCHEDULED_JOB")
    .otherwise("SYSTEM")
)

df = df.withColumn(
    "storage_type",
    when(col("source_path").startswith("s3://"), "AWS_S3")
    .when(col("source_path").startswith("abfss://"), "AZURE_ADLS")
    .when(col("source_path").startswith("gs://"), "GCP_GCS")
    .when(col("source_path").startswith("/Volumes/"), "UC_VOLUME")
    .otherwise("INTERNAL")
)


df = df.withColumn(
    "source_object_type",
    when(col("source_type") == "PATH", "FILE")
    .otherwise("TABLE")
)

df = df.filter(
    col("event_timestamp").isNotNull()
)
df = df.dropDuplicates([
    "record_id",
    "event_id",
    "source_object",
    "target_object"
])



df = df.drop(
    "entity_metadata",
    "event_time"
)


In [0]:

df.write\
 .format("delta")\
 .mode("append") \
 .partitionBy("event_year", "event_month", "event_day") \
 .saveAsTable(tgt_silver_table)

In [0]:
%sql
select count(*) from data_governance.silver_lineage_analysis.lineage_table_level group by load_timestamp;